# اليوم الأول: ابنِ خط أساس وقارن التعزيز
**SDA-DSC-211 · Tamweel Lite · Day 1 · 60 minutes**  
إعداد وتقديم **ميعاد المري | Meaad Al-Marri**

ستدرّب Logistic Regression وXGBoost وLightGBM، وتقارن جودة ترتيب المخاطر وزمن التدريب، ثم تختار مرشحًا أوليًا بثلاث ملاحظات من نتائجك. لا يوجد نموذج مطلوب أن يفوز.

**تسليمك:** `day1_model_comparison.csv`، ملخص تشغيل، رسوم المقارنة، وقرارك المكتوب. شغّل الدفتر من البداية على **Colab CPU المجاني**؛ لا تحتاج إلى تشغيل دفتر00 قبله أو ربط Drive. اترك الخانات الفارغة في أول تشغيل، ثم عدّلها وأعد خلية الحفظ.

**حدود هذا التطبيق:** التقسيم العشوائي الطبقي هنا نقطة مقارنة تعليمية مؤقتة. لا يفصل العملاء المتكررين ولا يحاكي ترتيب وصول الطلبات ونضج نتائجها. لذلك لا تستخدم أرقامه كتقييم نهائي؛ ستبني تحققًا صادقًا في اليوم الثاني. بيانات التحدي لا تُستخدم اليوم.

In this lab you will compare three models on the same development and comparison rows. This random split is a teaching baseline, not a deployment estimate. Save your own evidence and interpretation.

| الوقت | مهمتك |
|---|---|
| 0–5 | جهّز بيئة CPU |
| 5–12 | اقرأ البيانات والتقسيم |
| 12–22 | درّب خط الأساس |
| 22–38 | اختَر عدد أشجار التعزيز ثم أعد التدريب |
| 38–48 | قارن المقاييس والمنحنيات |
| 48–55 | اختر مرشحًا واكتب ثلاث ملاحظات |
| 55–60 | احفظ ملفاتك وأجب عن سؤالي الخروج |

## 1. جهّز بيئتك | Setup
اختر **Runtime → Run all**. يستخدم الدفتر بيانات وإصدارات ثابتة ببصمات SHA256. إذا طلبت رسالة الإعداد إعادة الجلسة، اختر **Restart session** ثم شغّل من البداية. لا يلزم اشتراك مدفوع.

الخيار الافتراضي `FAST_MODE=True`: حتى300 شجرة وخيطَي CPU. `FULL_MODE=True` يرفع الحد إلى900 شجرة عند الحاجة، بعد أن تحفظ تجربة البداية. كلاهما مجاني؛ الزمن الفعلي يتغير حسب الجهاز.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
LAB_REVISION = "c8344f6b19b26dcbce3dcfc671b746c59bf55895"
LAB_SHA256 = "2a968ba9e738cc5e3dae12077e576fdb40ec6ec64230613e445bbc7884e7e1ba"
fetch_verified(
    f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/scripts/day1_lab.py",
    ROOT / "scripts/day1_lab.py", LAB_SHA256)
print("Day 1 setup verified | إعداد اليوم الأول جاهز")

## 2. اقرأ المهمة والبيانات | Data
كل السجلات **اصطناعية**. الهدف `default_within_90d` يرصد حدث تعثر خلال90 يومًا بعد الطلب، وليس تأخرًا بلغ90 يومًا. نتنبأ من22 خاصية متاحة عند الطلب؛ نستبعد المعرّفات والتاريخ والهدف من المدخلات. بعض القيم مفقودة، وسنتعلم التعويض من جزء التدريب فقط.

اكتب في نهاية التطبيق: ما النتيجة التي تقدّرها، متى تتخذ القرار، وما حدود استخدامه؟ لا تتخذ قرارات تمويل حقيقية من هذه المحاكاة.

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_curve, precision_recall_curve
from data_checks import load_course_data
from day1_lab import (LabConfig, MODEL_NAMES, make_splits, select_tree_count,
                      fit_model, compare_models, learner_checkpoint, config_dict)

warnings.filterwarnings("ignore", message="scipy.optimize:.*disp.*", category=DeprecationWarning)
frames, contract, data_summary = load_course_data(ROOT)
train = frames["train"]
FAST_MODE, FULL_MODE = True, False  # Choose exactly one mode.
assert FAST_MODE != FULL_MODE, "Choose exactly one mode."
config = LabConfig(max_trees=300 if FAST_MODE else 900)
X, y, splits = make_splits(train, contract, seed=config.seed)
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
display(pd.DataFrame({"rows": [len(train)], "predictors": [X.shape[1]],
                      "positive_rate": [y.mean()], "missing_cells": [int(X.isna().sum().sum())]}))
display(train[["application_id", "income_sar", "bureau_score", contract["target"]["name"]]].head(3))

## 3. افصل الأدوار قبل التدريب | Three roles
من10,000 طلب نحتفظ بـ2,000 للمقارنة، و8,000 للتطوير. نقسم التطوير داخليًا إلى6,000 لتعلّم الأشجار و2,000 للإيقاف المبكر. نتعلم تعويض المفقود من6,000 فقط أثناء اختيار عدد الأشجار.

بعد تثبيت العدد، نعيد تعلّم التعويض وتدريب كل نموذج على8,000 طلب تطوير. بعدها فقط نحسب نتائج المقارنة على2,000 طلب لم تدخل في التدريب أو اختيار عدد الأشجار. هذا يفصل دور المقارنة عن الإيقاف المبكر، لكنه **لا يعالج** تسرب العميل أو الزمن في التقسيم العشوائي.

The inner stopping set selects a tree count. The outer comparison set is scored only after fitting. Repeated inspection and tuning against the comparison scores would turn it into a tuning set.

In [ ]:
development, comparison = splits["development"], splits["comparison"]
inner_fit, inner_stop = splits["inner_fit"], splits["inner_stop"]
assert not (set(development) & set(comparison))
assert not (set(inner_fit) & set(inner_stop))
split_table = pd.DataFrame([
    {"role": role, "rows": len(index), "positive_rate": y.loc[index].mean()}
    for role, index in splits.items()
])
shared_customers = len(set(train.loc[development, "customer_id"]) & set(train.loc[comparison, "customer_id"]))
display(split_table)
print(f"Customers shared across development/comparison: {shared_customers} — revisit on Day 2")
models, timings, selections = {}, {}, {}

## 4. ابدأ بمرجع بسيط | Logistic Regression
التعويض بالوسيط ثم توحيد المقاييس ثم Logistic Regression في Pipeline واحدة. تتعلم الخطوات كلها من التطوير فقط. هذا خط أساس يستحق المقارنة؛ لا تفترض أن زيادة التعقيد تحسّن النتيجة.

In [ ]:
name = "Logistic Regression"
models[name], timings[name] = fit_model(name, X.loc[development], y.loc[development], config)
print(f"Baseline fitted on {len(development):,} rows in {timings[name]['train_seconds']:.3f} s")

## 5. جرّب التعزيز والإيقاف المبكر | Boosting
يضيف التعزيز أشجارًا بالتتابع لتقليل دالة الخسارة؛ في التصنيف اللوجستي تتراكم الإضافات في فضاء log-odds قبل تحويلها إلى احتمال. ليس الأمر جمع احتمالات الأشجار مباشرة.

| المفتاح | ما تراقبه |
|---|---|
| `learning_rate` | حجم مساهمة كل شجرة؛ الأصغر قد يحتاج أشجارًا أكثر |
| `max_trees` | سقف البحث عن عدد الأشجار، وليس عددًا ملزمًا |
| `xgb_depth` / `lgb_leaves` | تعقيد الأشجار؛ نمو XGBoost الافتراضي هنا depth-wise وLightGBM leaf-wise |
| `min_child_samples` | يمنع أوراق LightGBM الصغيرة جدًا |
| `patience` | عدد الجولات دون تحسن في **inner stop log loss** |

الإعدادات أعلاه نقطة بداية محدودة علىCPU. لا نضيف أوزان فئات اليوم؛ ستدرس أثرها لاحقًا. اقرأ الدوال في [day1_lab.py](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/scripts/day1_lab.py): التعويض الداخلي، اختيار الأشجار، ثم إعادة التدريب واضحة ومنفصلة.

`selected_trees` عدد يبدأ من1؛ لذلك نحوّل فهرس XGBoost `best_iteration` بإضافة1. LR ليست له أشجار وتظهر خانته فارغة.

In [ ]:
for name in ("XGBoost", "LightGBM"):
    selections[name] = select_tree_count(
        name, X.loc[inner_fit], y.loc[inner_fit], X.loc[inner_stop], y.loc[inner_stop], config)
    models[name], timings[name] = fit_model(
        name, X.loc[development], y.loc[development], config, selections[name])
    print(f"{name}: {selections[name]['selected_trees']} selected trees; "
          f"{selections[name]['rounds_evaluated']} evaluated rounds; "
          f"{timings[name]['train_seconds']:.3f} s including selection + refit")
    if selections[name]["hit_tree_budget"]:
        print("Tree budget reached: record this limitation before considering FULL_MODE.")

### راقب منحنى التعلّم | Learning curves
الخط المتقطع يحدد عدد الأشجار المختار من المجموعة الداخلية. انخفاض خسارة التدريب وحدها لا يكفي. سجّل كيف تتغير خسارة الإيقاف، وهل وصلت إلى سقف الأشجار. الرسوم تستخدم عناوين إنجليزية قصيرة لتسهيل قراءتها في مختلف البيئات.

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
colors = {"Logistic Regression": "#2667A0", "XGBoost": "#BA5A17", "LightGBM": "#18816A"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, (name, result) in zip(axes, selections.items()):
    rounds = np.arange(1, len(result["curves"]["fit"]) + 1)
    ax.plot(rounds, result["curves"]["fit"], label="Inner fit (6,000)", color="#2667A0")
    ax.plot(rounds, result["curves"]["stop"], label="Inner stop (2,000)", color="#BA5A17", linestyle="--")
    ax.axvline(result["selected_trees"], color="#444444", linestyle=":",
               label=f"Selected: {result['selected_trees']} trees")
    ax.set(title=name, xlabel="Boosting round (1-based)", ylabel="Log loss (lower is better)")
    ax.legend(fontsize=9)
fig.suptitle("Tree count selected using inner rows only")
fig.savefig(ARTIFACTS / "day1_learning_curves.png", dpi=160)
plt.show()

## 6. قارن النماذج على الصفوف نفسها | Comparison
**ROC-AUC** يلخص ترتيب الموجب مقابل السالب عبر العتبات. **Average Precision (AP)** يلخص منحنى precision–recall وهو مفيد عند ندرة الموجب. نستخدم AP عندما نقول PR-AUC في هذا التطبيق؛ ليس مساحة شبه المنحرف. أبلغ عن المقياسين مع نسبة الموجب وحجم العينة.

`train_seconds` يشمل اختيار عدد الأشجار وإعادة التدريب للمعززات، والتدريب فقط للمرجع؛ لا يشمل تنزيل الحزم أو رسم المنحنيات. يظهر أيضًا زمن كل جزء وزمن التنبؤ. الزمن رصد لهذا التشغيل، وليس ضمانًا على جهاز آخر.

هذه تجربة واحدة بلا فترات ثقة، ونتائجها لا تثبت تفوقًا عامًا أو معايرة جيدة أو فائدة تشغيلية. الخسارة الافتراضية `10×FN + FP` وسعة12% ستدخل اختيار القرار في تطبيق لاحق.

In [ ]:
comparison_table, probabilities = compare_models(
    models, timings, selections, X.loc[comparison], y.loc[comparison])
comparison_table["selected_trees"] = comparison_table["selected_trees"].astype("Int64")
display(comparison_table[["model", "roc_auc", "average_precision", "train_seconds",
                          "selected_trees", "comparison_rows", "positive_rate"]].round(4))
comparison_table.to_csv(ARTIFACTS / "day1_model_comparison.csv", index=False)
prediction_export = train.loc[comparison, ["application_id", "default_within_90d"]].join(probabilities)
prediction_export.to_csv(ARTIFACTS / "day1_comparison_predictions.csv", index=False)
assert len(comparison_table) == 3
assert comparison_table[["roc_auc", "average_precision", "train_seconds"]].notna().all().all()
print("TECHNICAL_READY — three models compared; your interpretation is still required.")

### اقرأ المنحنيين معًا | ROC and precision–recall
الخط المرجعي في ROC يساوي ترتيبًا عشوائيًا. خط precision المرجعي يساوي نسبة الموجب في عينة المقارنة. قارن قرب المنحنيات وتغيّر precision عند recall مرتفع، ثم ارجع إلى الجدول للأرقام الدقيقة.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
styles = ["-", "--", "-."]
for name, style in zip(MODEL_NAMES, styles):
    fpr, tpr, _ = roc_curve(y.loc[comparison], probabilities[name])
    precision, recall, _ = precision_recall_curve(y.loc[comparison], probabilities[name])
    row = comparison_table.set_index("model").loc[name]
    axes[0].plot(fpr, tpr, color=colors[name], linestyle=style, label=f"{name}: {row.roc_auc:.3f}")
    axes[1].plot(recall, precision, color=colors[name], linestyle=style, label=f"{name}: AP {row.average_precision:.3f}")
axes[0].plot([0, 1], [0, 1], color="#777777", linestyle=":", label="Random ranking")
axes[1].axhline(y.loc[comparison].mean(), color="#777777", linestyle=":",
               label=f"Prevalence: {y.loc[comparison].mean():.1%}")
axes[0].set(title="ROC", xlabel="False positive rate", ylabel="True positive rate")
axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
axes[0].legend(fontsize=8, loc="lower right")
axes[1].legend(fontsize=8, loc="upper right")
fig.suptitle(f"Teaching comparison · {len(comparison):,} synthetic requests · random split")
fig.savefig(ARTIFACTS / "day1_roc_pr.png", dpi=160)
plt.show()

## 7. اختر مرشحًا واكتب حجتك | Your decision
عدّل الخانات التالية بكلماتك. **لا يوجد اختيار تلقائي ولا إجابة جاهزة.**

1. **الدليل:** اذكر رقمًا أو مقارنة من نتائجك تدعم اختيارك.
2. **القيد:** ما الذي لا تستطيع هذه التجربة إثباته؟
3. **التجربة التالية:** ما الاختبار الذي قد يغير اختيارك؟

**سؤالا الخروج:** لماذا تنظر إلىAP بجانبROC-AUC عندما يكون الموجب نادرًا؟ ولماذا لا تستخدم عينة المقارنة لاختيار لحظة الإيقاف المبكر؟

Write a short problem statement, one candidate, three evidence-based notes and two exit answers. The check below only detects missing fields; it does not assess correctness or calculate a grade.

In [ ]:
problem_statement = ""  # Outcome, decision time and intended teaching use.
candidate = None  # "Logistic Regression", "XGBoost", or "LightGBM"
notes = {"evidence": "", "limitation": "", "next_test": ""}
exit_answers = ["", ""]
checkpoint = learner_checkpoint(candidate, notes, problem_statement, exit_answers)
print(checkpoint["status"])
for reminder in checkpoint["missing"]:
    print("-", reminder)

## 8. احفظ أدلتك | Export
تعاد كتابة مخرجات اليوم الأول عند تشغيل الخلايا. نزّل الحزمة قبل تغيير الإعدادات إذا أردت الاحتفاظ بتجربتين. الخلية تحفظ CSV والرسوم والإعدادات وتوزيع الصفوف وملاحظاتك وبيئة التشغيل في `day1_artifacts.zip`.

من **Files → tamweel → artifacts** نزّل ZIP ثم افكّه وارفع الملفات إلى `artifacts/` في مستودعك. احفظ Notebook من **File → Download → Download .ipynb** وارفعه إلى `notebooks/`. انسخ صياغة المشكلة من ملاحظاتك إلىREADME مشروعك. لا ترفع رمز دخول أو بيانات شخصية.

إذا ظهر `LEARNER_WORK_REQUIRED` فالتشغيل التقني نجح لكن بقيت إجاباتك. `READY_FOR_REVIEW` يعني أن الخانات مكتملة فقط، ولا يمنح درجة أو يضمن صحة التفسير.

In [ ]:
import platform, zipfile

checkpoint = learner_checkpoint(candidate, notes, problem_statement, exit_answers)
reflection = {"problem_statement": problem_statement, "candidate": candidate,
              "notes": notes, "exit_answers": exit_answers, "checkpoint": checkpoint}
(ARTIFACTS / "day1_reflection.json").write_text(json.dumps(reflection, ensure_ascii=False, indent=2), encoding="utf-8")
membership = train[["application_id"]].copy()
membership["role"] = "comparison"
membership.loc[inner_fit, "role"] = "inner_fit"
membership.loc[inner_stop, "role"] = "inner_stop"
membership.to_csv(ARTIFACTS / "day1_split_membership.csv", index=False)
report = {"technical_status": "TECHNICAL_READY", "learner_status": checkpoint["status"],
          "data_revision": COURSE_REVISION, "lab_revision": LAB_REVISION,
          "lab_sha256": LAB_SHA256, "data_manifest_sha256": MANIFEST_SHA256,
          "training_file_sha256": hashlib.sha256((ROOT / "data/tamweel_train.csv").read_bytes()).hexdigest(),
          "config": config_dict(config), "mode": "FAST" if FAST_MODE else "FULL",
          "python": platform.python_version(), "split": split_table.to_dict("records"),
          "split_method": "random stratified teaching baseline; not final validation",
          "shared_customers": shared_customers, "selections": selections,
          "ap_definition": "sklearn average_precision_score, non-interpolated",
          "elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 2)}
if IN_COLAB:
    import resource
    report["peak_process_memory_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
(ARTIFACTS / "day1_run.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
export_names = ["environment.json", "day1_model_comparison.csv", "day1_comparison_predictions.csv",
                "day1_split_membership.csv", "day1_learning_curves.png", "day1_roc_pr.png",
                "day1_reflection.json", "day1_run.json"]
with zipfile.ZipFile(ARTIFACTS / "day1_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in export_names:
        bundle.write(ARTIFACTS / filename, filename)
print(f"TECHNICAL_READY | {checkpoint['status']} | {report['elapsed_seconds']} seconds | CPU")
print("Saved day1_artifacts.zip — download it and save your notebook.")

## قبل أن تغلق | Check and continue
- [ ] ظهرت ثلاثة صفوف بمقاييس صالحة في `day1_model_comparison.csv`.
- [ ] شرحت الفرق بين التطوير والإيقاف الداخلي والمقارنة.
- [ ] اخترت مرشحًا وكتبت دليلًا وقيدًا وتجربة تالية.
- [ ] كتبت صياغة المشكلة وأجبت عن سؤالي الخروج وحفظت الملفات.

**توسّع اختياري:** احتفظ بتجربتك الأساسية ثم غيّر `learning_rate` مرة واحدة مع التقسيم نفسه. فسّر أثره على عدد الأشجار والزمن. المقارنة المتكررة ليست اختبارًا نهائيًا؛ وثّق كل محاولة. لا تُكثر البحث عن أعلى رقم اليوم.

**عند التعثر:** فشل التنزيل → أعد خلية الإعداد؛ اختلاف إصدار مستورد → Restart session ثم Run all؛ اختلاف البصمة → استرجع الملفات الأصلية دون تجاوز الفحص؛ انقطاع الجلسة → افتح نسختك المحفوظة وأعد التشغيل. لا تستبدل نتائجك بجدول شخص آخر. [دليل اليوم الأول](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/DAY1_GUIDE.md).

**في اليوم الثاني:** اختبر أثر فصل العملاء والزمن ونضج الهدف، وراجع ما إذا بقي مرشحك الأول مناسبًا.

مراجع: [AP في scikit-learn1.6](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.average_precision_score.html) · [الإيقاف المبكر في XGBoost](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html) · [الإيقاف المبكر في LightGBM4.6](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.early_stopping.html).